# Insert sample orders into Azure SQL (Exercise 1b)

This notebook inserts **5 new rows** into `[SalesLT].[SalesOrderHeader]` in the workshop Azure SQL Database so the Copy Job has new data to pick up on its next incremental run.

It connects to Azure SQL as the **currently signed-in user** (your Entra identity) using an access token — no SQL username/password and no local SQL client required.

Because everyone shares one database, each attendee's rows are tagged in the **`Comment`** column with **`SUSER_NAME()`** — your own SQL user name. That makes your five rows easy to select back (the verification query filters on `SUSER_NAME()`), and `SalesOrderID` is left to auto-increment so there are no primary-key conflicts.

**Prerequisites**

- The `ODBC Driver 18 for SQL Server` is preinstalled in the Fabric Spark runtime.
- Setup gave the workshop participants group (which contains your account) read and write access to `sqldb-di-workshop`.
- Your workspace has a managed private endpoint to the SQL server (it has no public endpoint), created by setup. The first Spark session can take a few minutes to start because the workspace uses a managed virtual network.

## Cell 1 — Configuration

In [ ]:
SQL_SERVER   = "sqldiws2026.database.windows.net"
SQL_DATABASE = "sqldb-di-workshop-diuserXX"
NUM_ROWS     = 5

## Cell 2 — Connect to Azure SQL as the signed-in user

Acquires an access token for the current user and opens a `pyodbc` connection with it.

In [ ]:
import struct
import pyodbc
import notebookutils

# Token for the currently signed-in user (your Entra identity)
access_token = notebookutils.credentials.getToken("https://database.windows.net/")
token_bytes  = access_token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
SQL_COPT_SS_ACCESS_TOKEN = 1256

conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
    f"Encrypt=yes;TrustServerCertificate=no;"
)

conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
print(f"Connected to {SQL_DATABASE} as the signed-in user.")

## Cell 3 — Insert 5 rows into SalesLT.SalesOrderHeader

`SalesOrderID` auto-increments; every row's `Comment` is set to `SUSER_NAME()` (your SQL user name).

In [ ]:
values = ",\n    ".join(
    "(GETUTCDATE(), DATEADD(DAY, 7, GETUTCDATE()), 1, 'CARGO TRANSPORT 5', SUSER_NAME(), GETUTCDATE())"
    for _ in range(NUM_ROWS)
)

insert_sql = f"""
SET NOCOUNT ON;
INSERT INTO SalesLT.SalesOrderHeader
    (OrderDate, DueDate, CustomerID, ShipMethod, Comment, ModifiedDate)
OUTPUT inserted.SalesOrderID, inserted.Comment
VALUES
    {values};
"""

cur = conn.cursor()
cur.execute(insert_sql)
rows = cur.fetchall()
conn.commit()

print(f"Inserted {len(rows)} rows into SalesLT.SalesOrderHeader:")
for r in rows:
    print(f"  SalesOrderID={r[0]}  Comment={r[1]}")

cur.close()
conn.close()